# Improving the precision and recall of the curation model (fig5e, spontaneous regime)

**purpose**: find out whether a different model, or additional features, raise precision AND recall of the "good" single-unit classifier substantially, relative to the current model (unregularized binomial GLM on 10 quality metrics, probability threshold 0.8).

**validation (nested cross-validation, to avoid fooling ourselves)**
- outer loop: 10 x repeated stratified 4-fold (75% train / 25% test, as in the paper) on the 182 units. The test units are never used to choose anything.
- inner loop (training units only): hyper-parameters AND the decision threshold are chosen by 5-fold cross-validation, maximizing the F1 score.
- every model is evaluated on the same outer folds, so differences to the baseline are paired.
- features use no ground truth: the 10 quality metrics, and 6 extra features computed from the sorted unit's template and waveforms.

Requires the waveforms saved by fig5e.ipynb.

* `avg_precision (PR-AUC)` is threshold-free: it says whether a model's precision-recall curve is higher, i.e. whether it could reach higher precision at the same recall.

In [1]:
%load_ext autoreload
%autoreload 2
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
import warnings
import numpy as np
import pandas as pd
import spikeinterface as si
import spikecurate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn import metrics as skm
warnings.filterwarnings("ignore")

os.chdir("/home/steeve/steeve/epfl/code/spikebias/")
from src.nodes.models.Flc import dataloader

EXP, SORTER = "NS", "KS4"
WE_SINGLE_U_PATH = "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m_single_units"
SORTING_TRUE_10mn = "dataset/00_raw/ground_truth_npx_spont_10m"
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"
WE_JOB_KWARGS = dict(n_jobs=8, chunk_duration="1s", progress_bar=False)
LABEL = "quality_label"

## Dataset: the 10 quality metrics (quality metrics are recomputed, ~1.5 min)

In [2]:
%%time
We = si.load_waveforms(WE_SINGLE_U_PATH, with_recording=True)
single_unit_ids = np.array([int(u) for u in We.unit_ids])
good_ids, bad_ids = dataloader.get_good_and_bad_units(QUALITY_PATH, exp=EXP, sorter=SORTER)
SortingTrue = si.load_extractor(SORTING_TRUE_10mn)
data = spikecurate.load_dataset(single_unit_ids, good_ids, bad_ids, We.sorting, SortingTrue, We,
                                load_qm_if_exists=False, delta_time=1.3, job_kwargs=WE_JOB_KWARGS)
base_predictors = list(data["predictors"])
print(data["dataset"].shape, base_predictors)

(184, 11) ['amplitude_cutoff', 'firing_range', 'firing_rate', 'isi_violations_ratio', 'rp_contamination', 'rp_violations', 'sd_ratio', 'snr', 'silhouette', 'mad_ratio']
CPU times: user 12.4 s, sys: 4.24 s, total: 16.7 s
Wall time: 1min 28s


## Extra features from the template and waveforms of each unit (no ground truth)

* `amp_template`: trough amplitude of the template on the main channel
* `width_ms`: trough-to-peak time
* `n_channels`: number of channels in the unit's footprint (sparsity)
* `trough_cv`: variability of the trough amplitude across the unit's spikes
* `shape_corr_med`, `shape_lowcorr`: median correlation of each spike's waveform with the template, and the fraction of spikes below 0.7 (heterogeneous waveforms point to merged units)
* `amp_median`: median spike amplitude (SpikeInterface)

In [3]:
%%time
from spikeinterface.core import get_template_extremum_channel
from spikeinterface.qualitymetrics.misc_metrics import compute_amplitude_medians

fs = We.sampling_frequency
ext_ch = get_template_extremum_channel(We, peak_sign="neg", outputs="index")
amp_med = compute_amplitude_medians(We)
rows = {}
for u in We.unit_ids:
    ch = ext_ch[u]
    inds = We.sparsity.unit_id_to_channel_indices[u]       # the unit's (sparse) channels
    pos = int(np.where(inds == ch)[0][0])                   # position of its main channel
    tmpl = We.get_template(u, mode="average")
    tmpl = tmpl[:, pos] if tmpl.shape[1] == len(inds) else tmpl[:, ch]
    trough = int(np.argmin(tmpl))
    peak = trough + int(np.argmax(tmpl[trough:]))
    w = We.get_waveforms(u)[:, :, pos]
    ta = w[:, trough]
    corr = np.array([np.corrcoef(wi, tmpl)[0, 1] for wi in w])
    rows[int(u)] = dict(amp_template=-tmpl[trough], width_ms=(peak - trough) / fs * 1000, n_channels=len(inds),
                        trough_cv=np.std(ta) / abs(np.mean(ta)), shape_corr_med=np.nanmedian(corr),
                        shape_lowcorr=np.nanmean(corr < 0.7), amp_median=amp_med[u])
extra = pd.DataFrame.from_dict(rows, orient="index")
extra_predictors = list(extra.columns)
dataset = data["dataset"].join(extra).dropna()          # drops the 2 units with a missing silhouette (182 units)
y = dataset[LABEL].values.astype(int)
print(dataset.shape, "good:", y.sum(), "bad:", (1 - y).sum())
display(dataset[extra_predictors].describe().round(3))

(182, 18) good: 84 bad: 98


,amp_template,width_ms,n_channels,trough_cv,shape_corr_med,shape_lowcorr,amp_median
count,182.000,182.000,182.000,182.000,182.000,182.000,182.000
mean,31.582,0.918,28.802,0.776,0.613,0.582,30.652
std,37.377,0.322,0.983,1.413,0.259,0.426,38.292
min,0.783,0.150,28.000,0.037,-0.032,0.000,2.221
25%,10.213,0.875,28.000,0.193,0.404,0.067,8.563
50%,18.252,0.925,28.000,0.365,0.636,0.800,17.658
75%,41.534,0.950,30.000,0.805,0.841,0.994,36.701
max,283.768,4.625,30.000,13.142,0.985,1.000,300.824


CPU times: user 1.91 s, sys: 150 ms, total: 2.06 s
Wall time: 2.06 s


## Nested cross-validation

In [4]:
OUTER = RepeatedStratifiedKFold(n_splits=4, n_repeats=10, random_state=0)   # 75% / 25%, 40 folds
INNER_SPLITS = 5

def scores(est, X):
    return est.predict_proba(X)[:, 1] if hasattr(est, "predict_proba") else est.decision_function(X)

def best_threshold(y_true, s):
    """threshold maximizing F1 on out-of-fold scores"""
    cand = np.unique(s)
    cand = (cand[:-1] + cand[1:]) / 2 if len(cand) > 1 else cand
    f1 = [skm.f1_score(y_true, (s >= t).astype(int), zero_division=0) for t in cand]
    k = int(np.argmax(f1)); return cand[k], f1[k]

def tuned(estimator, grid, Xtr, ytr, seed):
    """choose hyper-parameters and threshold by inner CV, refit on all the training units"""
    skf = StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=seed)
    best = None
    for params in ParameterGrid(grid):
        oof = np.zeros(len(ytr))
        for a, b in skf.split(Xtr, ytr):
            m = clone(estimator).set_params(**params).fit(Xtr[a], ytr[a]); oof[b] = scores(m, Xtr[b])
        t, f1 = best_threshold(ytr, oof)
        if best is None or f1 > best[0]: best = (f1, params, t)
    _, params, t = best
    return clone(estimator).set_params(**params).fit(Xtr, ytr), t

def logit(**kw): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, **kw))
MODELS = {
    "logit + tuned thr":   (logit(penalty=None), {}),
    "L1 logit":            (logit(penalty="l1", solver="liblinear"), {"logisticregression__C": [0.1, 0.3, 1, 3, 10]}),
    "L2 logit":            (logit(penalty="l2"), {"logisticregression__C": [0.03, 0.1, 0.3, 1, 3]}),
    "gradient boosting":   (GradientBoostingClassifier(random_state=0), {"n_estimators": [100, 300], "max_depth": [1, 2, 3], "learning_rate": [0.05]}),
    "random forest":       (RandomForestClassifier(n_estimators=300, random_state=0), {"min_samples_leaf": [1, 3, 5]}),
    "extra trees":         (ExtraTreesClassifier(n_estimators=300, random_state=0), {"min_samples_leaf": [1, 3, 5]}),
    "SVM (RBF)":           (make_pipeline(StandardScaler(), SVC()), {"svc__C": [0.3, 1, 3, 10], "svc__gamma": ["scale", 0.03]}),
}

def one_fold(k, tr, te, cols, seed=0):
    X = dataset[cols].values
    out = {}
    # baseline: unpenalized logistic regression, fixed threshold 0.8 (current model)
    base, _ = tuned(logit(penalty=None), {}, X[tr], y[tr], seed)
    pred = (scores(base, X[te]) >= 0.8).astype(int)
    out["baseline (GLM, thr 0.8)"] = (skm.precision_score(y[te], pred, zero_division=0), skm.recall_score(y[te], pred, zero_division=0),
                                      skm.average_precision_score(y[te], scores(base, X[te])))
    for name, (est, grid) in MODELS.items():
        m, t = tuned(est, grid, X[tr], y[tr], seed + k)
        pred = (scores(m, X[te]) >= t).astype(int)
        out[name] = (skm.precision_score(y[te], pred, zero_division=0), skm.recall_score(y[te], pred, zero_division=0),
                     skm.average_precision_score(y[te], scores(m, X[te])))
    return out

def run(cols):
    folds = list(OUTER.split(np.zeros(len(y)), y))
    res = Parallel(n_jobs=16)(delayed(one_fold)(k, tr, te, cols) for k, (tr, te) in enumerate(folds))
    names = list(res[0])
    return {n: np.array([r[n] for r in res]) for n in names}      # (folds, [precision, recall])

def summarize(res):
    base = res["baseline (GLM, thr 0.8)"]
    rows = []
    for n, a in res.items():
        p, r = a[:, 0], a[:, 1]; f1 = 2 * p * r / np.maximum(p + r, 1e-9)
        d = a - base
        ci = lambda x: 1.96 * np.std(x) / np.sqrt(len(x))
        rows.append({"model": n, "precision": np.median(p), "recall": np.median(r), "F1": np.median(f1),
                     "dPrecision": d[:, 0].mean(), "dP_ci": ci(d[:, 0]), "dRecall": d[:, 1].mean(), "dR_ci": ci(d[:, 1]),
                     "avg_precision (PR-AUC)": np.median(a[:, 2]), "dAP": d[:, 2].mean(), "dAP_ci": ci(d[:, 2])})
    t = pd.DataFrame(rows).set_index("model")
    t["both improve"] = (t["dPrecision"] - t["dP_ci"] > 0) & (t["dRecall"] - t["dR_ci"] > 0)
    return t.round(3)

### A. the 10 quality metrics only

In [5]:
%%time
res_A = run(base_predictors)
summary_A = summarize(res_A); display(summary_A)

,precision,recall,F1,dPrecision,dP_ci,dRecall,dR_ci,avg_precision (PR-AUC),dAP,dAP_ci,both improve
model,,,,,,,,,,,
"baseline (GLM, thr 0.8)",0.879,0.619,0.706,0.000,0.000,0.000,0.000,0.895,0.000,0.000,False
logit + tuned thr,0.745,0.905,0.792,-0.147,0.022,0.261,0.032,0.895,0.000,0.000,False
L1 logit,0.731,0.905,0.816,-0.154,0.022,0.312,0.032,0.903,0.007,0.008,False
L2 logit,0.731,0.905,0.809,-0.152,0.021,0.304,0.028,0.892,-0.005,0.009,False
gradient boosting,0.792,0.905,0.833,-0.095,0.031,0.265,0.029,0.902,0.002,0.018,False
random forest,0.825,0.881,0.833,-0.072,0.029,0.263,0.035,0.916,0.014,0.018,False
extra trees,0.805,0.905,0.848,-0.059,0.025,0.280,0.031,0.932,0.028,0.014,False
SVM (RBF),0.800,0.905,0.830,-0.092,0.024,0.267,0.027,0.911,0.016,0.012,False


CPU times: user 150 ms, sys: 209 ms, total: 359 ms
Wall time: 32.2 s


### B. the 10 quality metrics + 6 template / waveform features

In [6]:
%%time
res_B = run(base_predictors + extra_predictors)
summary_B = summarize(res_B); display(summary_B)

,precision,recall,F1,dPrecision,dP_ci,dRecall,dR_ci,avg_precision (PR-AUC),dAP,dAP_ci,both improve
model,,,,,,,,,,,
"baseline (GLM, thr 0.8)",0.862,0.667,0.730,0.000,0.000,0.000,0.000,0.892,0.000,0.000,False
logit + tuned thr,0.736,0.857,0.804,-0.119,0.025,0.236,0.031,0.892,0.000,0.000,False
L1 logit,0.755,0.905,0.816,-0.099,0.023,0.256,0.031,0.894,0.002,0.014,False
L2 logit,0.750,0.905,0.824,-0.103,0.022,0.261,0.032,0.880,-0.012,0.013,False
gradient boosting,0.762,0.881,0.812,-0.095,0.025,0.229,0.039,0.896,-0.004,0.023,False
random forest,0.769,0.881,0.804,-0.087,0.032,0.218,0.041,0.903,0.014,0.022,False
extra trees,0.750,0.905,0.800,-0.094,0.029,0.219,0.039,0.901,0.015,0.018,False
SVM (RBF),0.800,0.857,0.821,-0.053,0.028,0.206,0.030,0.897,0.016,0.019,False


CPU times: user 147 ms, sys: 27 ms, total: 174 ms
Wall time: 34.8 s
